In [ ]:
def retrieve(query, fail=False):
        # fail=False 預設狀態為成功, 如使用 fail=True 則強制產生錯誤, 可做為測試用
    with tracer.start_as_current_span("retrieving_documents") as span:
            # 要求主機建立一個名為 "retrieving_documents" 的 span
        span.add_event("Starting retrieve")
        span.set_attribute("input.query", query)

        try:
            if fail: # 意義等於 if fail==True:
                raise ValueError(f"Retrieve failed for query: {query}")
                    # ValueError 錯誤類型: 傳入的參數內容不符合預期
                    # raise 表示會跳到 except Exception as e:

            retrieved_docs = ['retrieved doc1', 'retrieved doc2', 'retrieved doc3']
                # 模擬 RAG 撈出的資料列表, 實際上線程式則需要呼叫模型或是呼叫 API, 實際情況可能如下:
                # results = collection.query(
                # query_texts=[query],
                # n_results=3
                # )
                # retrieved_docs = results['documents'][0]
            for i, doc in enumerate(retrieved_docs): # 編號每個文件
                span.set_attribute(f"retrieval.documents.{i}.document.id", i)
                span.set_attribute(f"retrieval.documents.{i}.document.content", doc)
                span.set_attribute(f"retrieval.documents.{i}.document.metadata", f"Metadata for document {i}")

        except Exception as e:
            span.set_status(Status(StatusCode.ERROR, str(e)))
            span.set_attribute("error.type", type(e).__name__)
                # 記錄錯誤類型
            span.set_attribute("error.message", str(e))
                # 紀錄錯誤敘述內容
            raise

        span.set_status(Status(StatusCode.OK))
        return retrieved_docs

In [ ]:
def format_documents(retrieved_docs): #串接retrieved_docs列表內的文件成單一字串
    with tracer.start_as_current_span("call_format_documents") as span:
            # 要求主機建立一個名為 "call_format_documents" 的 span
        span.add_event("Calling format_documents")
            # 在 "call_format_documents" 這個 span 的時間軸上建立名為 "Calling format_documents" 的事件點 (event)
        span.set_attribute("input.documents_count", len(retrieved_docs))
            # 紀錄來自 retrieved_docs 下的文件數量
            # 在 "call_format_documents" 這個 span 的外層貼上紀錄 "input.documents_count" 的標籤

        t = '' # 空字串
        for i, doc in enumerate(retrieved_docs):
            t += f'Retrieved doc: {doc}\n'
            span.add_event(f"processed document {i}", {"document.content": doc})
                # 每處理完一份 doc 就在時間軸上建立一個 event, 同時文件也存入 event 中紀錄

        span.set_status(Status(StatusCode.OK)) # 回報主機執行成功
    return t

In [ ]:
def augment_prompt(query, formatted_documents): # 強化 prompt : query(user) + format_documents
    with tracer.start_as_current_span("augment_prompt") as span:
        span.add_event("Starting prompt augmentation")
        span.set_attribute("input.query", query)
        span.set_attribute("input.formatted_documents_length", len(formatted_documents))
            # 紀錄 formatted_documents 資料長度

        PROMPT = f"Answer the query: {query}.\nRelevant documents:\n{formatted_documents}"

        span.set_status(Status(StatusCode.OK))
    return PROMPT

In [ ]:
def generate(prompt):
    with tracer.start_as_current_span("generate") as span:
        span.add_event("Starting text generation")
        span.set_attribute("input.prompt", prompt)

        generated_text = f"Generated text for prompt {prompt}"
            # 模擬 LLM 回復
            # 實際狀況接近: response = openai.chat.completions.create(model="gpt-4",
                                                        # messages=[{"role": "user", "content": prompt}]
                                                        # )
                        # generate_text = response.choices[0].message.content

        span.set_status(Status(StatusCode.OK))
    return generated_text

In [ ]:
def rag_pipeline(query, fail = False): # main pipeline # query: user input
    with tracer.start_as_current_span("rag_pipeline") as span:
        try:
            retrieved_docs = retrieve(query, fail = fail)
                # 透過 def retrieve() 呼叫背後的RAG向量資料庫功能搜尋並取出文件
            formatted_docs = format_documents(retrieved_docs)
            prompt = augment_prompt(query, formatted_docs)
            generated_response = generate(prompt)

            span.set_status(Status(StatusCode.OK))
            return generated_response

        except Exception as e:
            span.set_status(Status(StatusCode.ERROR, str(e)))
            raise

In [ ]:
# test example
response = rag_pipeline("This is a test query", fail = False)